# `beta_activation_lmm.ipynb`\n\n**Purpose:** Test whether cortical activation (first-level GLM beta) changes\ndifferently across sessions between accelerated and non-accelerated groups.\n\n**Inputs:**\n- subjstats_completo.xlsx — fNIRS GLM betas from MATLAB SubjStats\n\n**Outputs:**\n- eta_lmm_{chromophore}.xlsx — LMM results (default: HbO)\n- Raincloud and trajectory plots (PNG)\n\n**Model:** eta ~ group * sessao_num + roi + (1|subject) [ML, BFGS]\n- Single model including both ROIs; roi as fixed covariate\n- N = 280 obs (14 subjects × 10 sessions × 2 ROIs; includes calibration)\n- Supports multiple chromophores via CHROMOPHORE variable (hbo, StO2, hbr, hbt)\n\n**Library:** statsmodels 0.14.6 (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Beta Activation LMM — Trajectory by Group and ROI
**Project SESI | Input: subjstats_all.xlsx**

Model: `beta ~ group * session + roi + (1|subject)`  
Session 0 = calibration (anchor/intercept)  
Sessions 1–9 = training trajectory  
Switch chromophore in Block 1 — everything propagates automatically.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration
**Change `CHROMOPHORE` to switch between `'hbo'`, `'StO2'`, `'hbr'`, `'hbt'`**

Excluir - 'S6-D5': ('P7-P9', 'L Inferior Temporal Gyrus (29%)', 'TEMPORAL'),

In [ ]:
# ── CHANGE THIS LINE TO SWITCH CHROMOPHORE ────────────────────
CHROMOPHORE = 'hbo'   # options: 'hbo' | 'StO2' | 'hbr' | 'hbt'
# ─────────────────────────────────────────────────────────────

# Update this path to match your local data directory
PATH_IN  = r'../data/subjstats_individuais.xlsx'

# Output paths auto-named by chromophore
# Update this path to match your local data directory
BASE_OUT = r'../results/beta_lmm'
PATH_OUT_EXCEL = f'{BASE_OUT}_{CHROMOPHORE}.xlsx'
PATH_OUT_TRAJ  = f'{BASE_OUT}_trajectory_{CHROMOPHORE}.png'
PATH_OUT_RAIN  = f'{BASE_OUT}_raincloud_{CHROMOPHORE}.png'
PATH_OUT_FOR   = f'{BASE_OUT}_forest_{CHROMOPHORE}.png'

# ROI channel lists
ROI_FRONTAL  = ['S1-D1','S2-D1','S2-D3','S1-D2','S3-D3','S4-D1','S3-D1']
ROI_TEMPORAL = ['S8-D7','S8-D4','S7-D7','S7-D4','S7-D6',
                'S6-D4','S5-D5','S5-D6','S5-D4']

PALETTE = {
    'acelerado'    : 'darkorange',
    'nao_acelerado': 'steelblue'
}
LABELS = {
    'acelerado'    : 'Accelerated',
    'nao_acelerado': 'Non-Accelerated'
}

print(f'Chromophore selected: {CHROMOPHORE}')
print(f'Output files will be saved with suffix: _{CHROMOPHORE}')

## 2 — Load and prepare

In [ ]:
df_raw = pd.read_excel(PATH_IN)

# Filter chromophore
df = df_raw[df_raw['type'] == CHROMOPHORE].copy()

# Extract session number from experiment string
# ac_01 → 1, ac_09 → 9, calibracao → 0
def parse_session(exp):
    exp = str(exp).strip().lower()
    if 'calib' in exp:
        return 0
    import re
    m = re.search(r'(\d+)', exp)
    return int(m.group(1)) if m else -1

df['sessao_num'] = df['experiment'].apply(parse_session)

# Assign ROI
def assign_roi(ch):
    if ch in ROI_FRONTAL:  return 'FRONTAL'
    if ch in ROI_TEMPORAL: return 'TEMPORAL'
    return 'OTHER'

df['roi'] = df['channel'].apply(assign_roi)
df = df[df['roi'] != 'OTHER'].copy()

# Aggregate beta by subject × session × roi
df_agg = (
    df.groupby(['subject', 'group', 'sessao_num', 'roi'])['beta']
    .mean()
    .reset_index()
)

# Set reference categories
df_agg['group'] = pd.Categorical(
    df_agg['group'],
    categories=['nao_acelerado', 'acelerado'],
    ordered=False
)
df_agg['roi'] = pd.Categorical(
    df_agg['roi'],
    categories=['FRONTAL', 'TEMPORAL'],
    ordered=False
)
df_agg['subject'] = pd.Categorical(df_agg['subject'])

print(f'Chromophore : {CHROMOPHORE}')
print(f'Shape       : {df_agg.shape}')
print(f'Subjects    : {df_agg["subject"].nunique()}  (expected 14)')
print(f'Sessions    : {sorted(df_agg["sessao_num"].unique())}')
print(f'ROIs        : {df_agg["roi"].unique().tolist()}')
print(f'Missing beta: {df_agg["beta"].isna().sum()}')
print()
print('Beta descriptive by group and ROI:')
print(df_agg.groupby(['group','roi'], observed=True)['beta']
      .describe().round(3))

## 3 — LMM: `beta ~ group * session + roi + (1|subject)`
Session 0 = calibration anchor

In [ ]:
FORMULA = 'beta ~ group * sessao_num + roi'

lmm = smf.mixedlm(FORMULA, data=df_agg, groups=df_agg['subject'])
res = lmm.fit(reml=False, method='bfgs')
print(res.summary())

# Clean results table
fe_index = res.fe_params.index
ci       = res.conf_int().loc[fe_index]
df_lmm = pd.DataFrame({
    'parameter': fe_index,
    'coef'     : res.fe_params.values,
    'se'       : res.bse.loc[fe_index].values,
    'z'        : (res.fe_params / res.bse.loc[fe_index]).values,
    'pvalue'   : res.pvalues.loc[fe_index].values,
    'ci_lower' : ci[0].values,
    'ci_upper' : ci[1].values,
}).assign(
    significant=lambda d: d['pvalue'] < 0.05,
    chromophore=CHROMOPHORE
)

PARAM_LABELS = {
    'Intercept'                      : 'Intercept (Non-Acc, Frontal, Session 0)',
    'group[T.acelerado]'             : 'Group Effect (Accelerated)',
    'sessao_num'                     : 'Session Progression',
    'roi[T.TEMPORAL]'                : 'ROI (Temporal vs Frontal)',
    'group[T.acelerado]:sessao_num'  : 'Training-Induced Beta Change ★',
}
df_lmm['label'] = df_lmm['parameter'].map(PARAM_LABELS).fillna(df_lmm['parameter'])

print('\nKey results:')
print(df_lmm[['label','coef','pvalue','ci_lower','ci_upper','significant']]
      .to_string(index=False))

## 4 — Forest plot

In [ ]:
df_plot = df_lmm[df_lmm['parameter'] != 'Intercept'].iloc[::-1].reset_index(drop=True)
color_main = 'seagreen'
colors = ['crimson' if '★' in str(l) and s else
          color_main if s else 'lightgrey'
          for l, s in zip(df_plot['label'], df_plot['significant'])]

fig, ax = plt.subplots(figsize=(12, 5))
for i, row in df_plot.iterrows():
    c = colors[i]
    ax.errorbar(
        y=i, x=row['coef'],
        xerr=[[row['coef']-row['ci_lower']], [row['ci_upper']-row['coef']]],
        fmt='o', color=c, capsize=5,
        markersize=9, markeredgecolor='black',
        ecolor=c, linewidth=1.5
    )
    p_str = f"p={row['pvalue']:.3f}" if row['pvalue'] >= 0.001 else 'p<0.001'
    ax.text(row['ci_upper'] + abs(row['ci_upper'])*0.05, i, p_str,
            va='center', fontsize=9,
            color='black' if row['significant'] else 'grey')

ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['label'], fontsize=11)
ax.axvline(0, linestyle='--', color='grey', linewidth=1.2)
ax.set_title(f'Beta Activation LMM — {CHROMOPHORE}',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Coefficient (β)', fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.5)

patches = [
    mpatches.Patch(color='crimson',   label='Key term p < 0.05'),
    mpatches.Patch(color=color_main,  label='p < 0.05'),
    mpatches.Patch(color='lightgrey', label='p ≥ 0.05'),
]
ax.legend(handles=patches, fontsize=9)
plt.tight_layout()
plt.savefig(PATH_OUT_FOR, dpi=300)
print(f'-> Figure saved: {PATH_OUT_FOR}')
plt.show()

## 5 — Trajectory plot by group and ROI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

roi_config = [
    ('FRONTAL',  axes[0], 'Frontal ROI'),
    ('TEMPORAL', axes[1], 'Temporal ROI'),
]

for roi, ax, subtitle in roi_config:
    data = df_agg[df_agg['roi'] == roi]
    traj = (
        data.groupby(['sessao_num', 'group'], observed=True)['beta']
        .agg(['mean', 'sem'])
        .reset_index()
    )
    for grp in ['acelerado', 'nao_acelerado']:
        sub   = traj[traj['group'] == grp].sort_values('sessao_num')
        color = PALETTE[grp]
        label = LABELS[grp]
        ax.plot(sub['sessao_num'], sub['mean'],
                marker='o', color=color, label=label, linewidth=2)
        ax.fill_between(
            sub['sessao_num'],
            sub['mean'] - sub['sem'],
            sub['mean'] + sub['sem'],
            alpha=0.2, color=color
        )

    # Mark calibration boundary
    ax.axvline(0.5, linestyle=':', color='grey', linewidth=1, alpha=0.6)
    ax.axhline(0, linestyle='--', color='grey', linewidth=0.8, alpha=0.5)
    ax.set_title(subtitle, fontsize=13, fontweight='bold')
    ax.set_xlabel('Session', fontsize=11)
    ax.set_ylabel(f'Mean Beta ({CHROMOPHORE}) ± SEM', fontsize=11)
    ax.set_xticks(range(0, 10))
    ax.set_xticklabels(['Calib'] + [str(i) for i in range(1, 10)])
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(fontsize=10)

fig.suptitle(f'Beta Activation Trajectory by Group and ROI — {CHROMOPHORE}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PATH_OUT_TRAJ, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_TRAJ}')
plt.show()

## 6 — Raincloud: mean beta per subject (sessions 1–9)

In [ ]:
# Mean beta per subject over training sessions only
df_train = df_agg[df_agg['sessao_num'] >= 1].copy()
df_subj = (
    df_train.groupby(['subject','group','roi'], observed=True)['beta']
    .mean()
    .reset_index()
    .rename(columns={'beta': 'beta_mean'})
)

# Mann-Whitney per ROI
mw_results = []
for roi in ['FRONTAL', 'TEMPORAL']:
    sub  = df_subj[df_subj['roi'] == roi]
    acc  = sub[sub['group'] == 'acelerado']['beta_mean'].values
    ctrl = sub[sub['group'] == 'nao_acelerado']['beta_mean'].values
    U, p = mannwhitneyu(acc, ctrl, alternative='two-sided')
    r    = 1 - (2*U)/(len(acc)*len(ctrl))
    mw_results.append({'roi': roi, 'U': U, 'p': round(p,4),
                       'r': round(r,3), 'sig': p < 0.05})
    print(f'{roi}: U={U}, p={p:.4f}, r={r:.3f} ({"✱" if p<0.05 else "n.s."})')

# Raincloud — 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=False)
roi_config = [('FRONTAL', axes[0], 'Frontal ROI'),
              ('TEMPORAL', axes[1], 'Temporal ROI')]

for roi, ax, subtitle in roi_config:
    sub = df_subj[df_subj['roi'] == roi]
    mw  = next(r for r in mw_results if r['roi'] == roi)

    for grp, xpos in [('acelerado', 0), ('nao_acelerado', 1)]:
        data  = sub[sub['group'] == grp]['beta_mean'].values
        color = PALETTE[grp]

        # Violin
        vp = ax.violinplot(data, positions=[xpos], widths=0.4,
                           showmeans=False, showmedians=False, showextrema=False)
        for body in vp['bodies']:
            m = np.mean(body.get_paths()[0].vertices[:, 0])
            if grp == 'acelerado':
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], m, np.inf)
            else:
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], -np.inf, m)
            body.set_facecolor(color)
            body.set_alpha(0.3)
            body.set_edgecolor(color)

        # Boxplot
        ax.boxplot(data, positions=[xpos], widths=0.12,
                   patch_artist=True, showfliers=False,
                   medianprops=dict(color='black', linewidth=2),
                   boxprops=dict(facecolor=color, alpha=0.6),
                   whiskerprops=dict(color=color),
                   capprops=dict(color=color))

        # Strip
        jitter = np.random.uniform(-0.06, 0.06, size=len(data))
        ax.scatter(xpos + jitter, data, color=color, alpha=0.9,
                   s=60, zorder=5, edgecolors='black', linewidths=0.5)

    # Stats annotation
    p_str = f"p={mw['p']:.3f}" if mw['p'] >= 0.001 else 'p<0.001'
    sig   = '✱' if mw['sig'] else 'n.s.'
    ax.annotate(
        f"{sig}  {p_str}\nr={mw['r']}",
        xy=(0.5, 0.97), xycoords='axes fraction',
        ha='center', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='grey', alpha=0.8)
    )
    ax.axhline(0, linestyle='--', color='grey', linewidth=0.8, alpha=0.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Accelerated', 'Non-Accelerated'], fontsize=11)
    ax.set_ylabel(f'Mean Beta ({CHROMOPHORE})', fontsize=11)
    ax.set_title(subtitle, fontsize=12, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle(f'Beta Activation by Group — {CHROMOPHORE}\n(Mean across training sessions)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PATH_OUT_RAIN, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_RAIN}')
plt.show()

## 7 — Export results

In [ ]:
mw_df = pd.DataFrame(mw_results)

with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_lmm.to_excel(writer,  sheet_name='LMM_results',    index=False)
    df_agg.to_excel(writer,  sheet_name='beta_long',       index=False)
    df_subj.to_excel(writer, sheet_name='subject_means',   index=False)
    mw_df.to_excel(writer,   sheet_name='mannwhitney',     index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print(f'   Chromophore: {CHROMOPHORE}')
print('   Sheets: LMM_results | beta_long | subject_means | mannwhitney')